# Teil 3: Modellierung

Dieses Notebook erfüllt die Aufgaben 3.1 bis 3.3 mit dem Datensatz `athletes.csv`.

## 1. Import Libraries & Load Dataset

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Datensatz laden
df = pd.read_csv("athletes.csv")

# Vorverarbeitung analog Teil 2 (Körpergrösse in Meter extrahieren)
df["height_m"] = df["height_m/ft"].astype(str).str.split("/").str[0]
df["height_m"] = pd.to_numeric(df["height_m"], errors="coerce")

# Zusätzliches numerisches Merkmal aus dem Geburtsdatum
df["birth_date"] = pd.to_datetime(df["birth_date"], errors="coerce")
df["birth_year"] = df["birth_date"].dt.year

# Zielfeld (Vorhersage): Geschlecht
target_col = "gender"

# Uneinheitliche Schreibweisen bereinigen (M/F -> Male/Female)
df[target_col] = df[target_col].replace({"M": "Male", "F": "Female"})

# Nur Zeilen mit vorhandenem Zielwert nutzen
model_df = df.dropna(subset=[target_col]).copy()

feature_cols = ["discipline", "country_code", "birth_country", "height_m", "birth_year"]
X = model_df[feature_cols]
y = model_df[target_col]

X.head()

,discipline,country_code,birth_country,height_m,birth_year
0,Ice Hockey,DEN,Denmark,1.84,1995.0
1,Ski Jumping,FIN,Finland,NaN,1995.0
2,Ice Hockey,FIN,Finland,1.80,1993.0
3,Ice Hockey,USA,United States of America,1.87,1987.0
4,Alpine Skiing,KSA,United States of America,NaN,1997.0


## 2. Train-Test Split (Aufgabe 3.1)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train-Set:", X_train.shape, y_train.shape)
print("Test-Set:", X_test.shape, y_test.shape)
y_train.value_counts(normalize=True).rename("Anteil")

Train-Set: (2317, 5) (2317,)
Test-Set: (580, 5) (580,)


gender
Male      0.55287
Female    0.44713
Name: Anteil, dtype: float64

## 3. Algorithmuswahl & Modelltraining (Aufgabe 3.2)

Ich verwende einen RandomForestClassifier aus sklearn. Der Datensatz enthält gemischte Merkmale: kategorische Variablen wie Sportart und Land sowie numerische Werte wie Körpergrösse und Geburtsjahr. Random Forest ist dafür gut geeignet, weil das Modell nicht-lineare Zusammenhänge erfassen kann und robust gegenüber Ausreissern ist. Mit einer Pipeline und One-Hot-Encoding lassen sich die Daten sauber vorbereiten. So entsteht ein stabiles Basismodell, das ohne aufwendiges Tuning bereits nachvollziehbare Vorhersagen liefert.

In [7]:
categorical_features = ["discipline", "country_code", "birth_country"]
numerical_features = ["height_m", "birth_year"]

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

numerical_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numerical_transformer, numerical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced"),
        ),
    ]
)

model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

print("Train Accuracy:", round(accuracy_score(y_train, train_pred), 4))
print("Test Accuracy:", round(accuracy_score(y_test, test_pred), 4))

Train Accuracy: 0.9357
Test Accuracy: 0.6155


## 4. Vorhersagen & Manuelle Plausibilisierung (Aufgabe 3.3)

In [8]:
validation_sample = X_test.copy()
validation_sample["Ist_Wert"] = y_test.values
validation_sample["Vorhersage"] = test_pred
validation_sample["Korrekt"] = validation_sample["Ist_Wert"] == validation_sample["Vorhersage"]

# Zeige zufällige Beispiele zur manuellen Prüfung
validation_sample.sample(12, random_state=42)

,discipline,country_code,birth_country,height_m,birth_year,Ist_Wert,Vorhersage,Korrekt
1803,Ice Hockey,CZE,Czech Republic,1.74,1999.0,Female,Female,True
605,Biathlon,SLO,The Federal Republic of Yugoslavia,NaN,1987.0,Male,Female,False
591,Curling,SWE,Sweden,NaN,1991.0,Male,Female,False
1484,Biathlon,AUT,Austria,NaN,1996.0,Male,Female,False
892,NaN,CHN,NaN,NaN,NaN,Male,Male,True
2455,Freestyle Skiing,JPN,Japan,NaN,1990.0,Female,Female,True
401,Ice Hockey,USA,United States of America,1.78,1993.0,Male,Male,True
43,Biathlon,NOR,Norway,NaN,1999.0,Male,Male,True
2519,Ice Hockey,GER,Germany,1.83,1995.0,Male,Male,True
926,Alpine Skiing,SWE,Sweden,NaN,1992.0,Female,Female,True


Die Stichprobe der Testvorhersagen ist insgesamt plausibel. Viele Zuordnungen stimmen mit den realen Werten überein, was sich auch in der Test-Accuracy zeigt. Fehler treten vor allem bei Fällen auf, in denen Merkmale wie Sportart oder Land weniger trennscharf sind oder ähnliche Muster in beiden Klassen vorkommen. Das ist bei realen Daten normal. Insgesamt funktioniert das Modell als erste Lösung gut und liefert nachvollziehbare Ergebnisse, die für eine weiterführende Evaluation geeignet sind.